In [ ]:
# =============================================================
# 🧠 Task 1: Wav2vec 2.0 Embedding Feature Extraction Pipeline
# =============================================================

# --- 1. Install Required Libraries ---
# Note: You may already have numpy and pandas installed from the previous notebook.
!pip install pandas numpy librosa torchaudio transformers tqdm openpyxl joblib

# --- 2. Import Libraries ---
import os
import pandas as pd
import numpy as np
import librosa
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import multiprocessing
import warnings
import random

# Suppress minor warnings for cleaner output
warnings.filterwarnings("ignore", category=UserWarning)

# --- 3. Define Paths and Constants ---

# IMPORTANT: Base this on the provided file path structure.
# Your base path for the 'Train' data is: C:\Users\vaish\OneDrive\Desktop\Sem 5\Multi Modal\Project\project2\Train\task1
# We'll use a relative path structure that aligns with the file image, assuming
# a starting point where the 'Train' directory is accessible.
# Since you provided an absolute path for the data location, let's use a dynamic approach
# based on the relative path structure *inside* your project.

# Adjust BASE_DIR according to your specific environment if needed.
# Based on your setup: BASE_DIR = r"C:\Users\vaish\OneDrive\Desktop\Sem 5\Multi Modal\Project\project2\Train\task1"
# For the notebook execution context, we'll keep the relative reference as in the previous notebook:
BASE_DIR = r"Train/task1" 
META_PATH = os.path.join(BASE_DIR, "sand_task_1.xlsx")
TRAIN_DIR = os.path.join(BASE_DIR, "training")
OUTPUT_DIR = BASE_DIR # Save output files in the same base directory

# --- Wav2vec 2.0 Configuration ---
# We use a robust, multilingual model (XLS-R) which performs well on Italian speech
# (the language of the SAND dataset) and typically requires 16kHz audio input.
MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
TARGET_SR = 16000 # Wav2vec 2.0 models typically expect 16kHz
EMBEDDING_DIM = 1024 # The output feature dimension of the XLS-R 300M model

# --- K-Fold Configuration ---
N_SPLITS = 5
RANDOM_STATE = 42

print(f"Base Directory Set to: {BASE_DIR}")
print(f"Wav2vec 2.0 Model: {MODEL_NAME} (Expected SR: {TARGET_SR}Hz, Dim: {EMBEDDING_DIM})")
print("-" * 70)

# Set seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     ------------------------------------- -- 41.0/44.0 kB 2.0 MB/s eta 0:00:01
     -------------------------------------- 44.0/44.0 kB 718.1 kB/s eta 0:00:00
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     -------------------------------------  41.0/41.5 kB 991.0 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 336.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/664.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/664.7 kB ? eta -:--:--
   -- ------------------------------------- 41.0/664.7 kB 1.9 MB/s eta 0:00:01
   --- ------------------------------------ 61.4/664.7 kB 1.1 MB/s eta 0:00:01
   ------- ------------------------------ 122.9/664.7 kB 901.1 kB/s eta 0:00:01
   -------- ----------------------------- 153.6/664.7 kB 833.5 kB/s eta 0:00:01
   ----------- -------------------------- 194.6/664.7 kB 841.6 kB/s eta 0:00


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\vaish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\vaish\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base Directory Set to: Train/task1
Wav2vec 2.0 Model: facebook/wav2vec2-xls-r-300m (Expected SR: 16000Hz, Dim: 1024)
----------------------------------------------------------------------


In [ ]:
!pip install protobuf


   ---------------------------------------- 0.0/436.9 kB ? eta -:--:--
    --------------------------------------- 10.2/436.9 kB ? eta -:--:--
   --- ----------------------------------- 41.0/436.9 kB 495.5 kB/s eta 0:00:01
   ----- --------------------------------- 61.4/436.9 kB 656.4 kB/s eta 0:00:01
   --------- ---------------------------- 112.6/436.9 kB 731.4 kB/s eta 0:00:01
   ------------- ------------------------ 153.6/436.9 kB 766.6 kB/s eta 0:00:01
   ------------------- ------------------ 225.3/436.9 kB 919.0 kB/s eta 0:00:01
   ---------------------------- ----------- 307.2/436.9 kB 1.1 MB/s eta 0:00:01
   ------------------------------------ --- 399.4/436.9 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 436.9/436.9 kB 1.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\vaish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

In [6]:
!pip install hf_xet

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB 320.0 kB/s eta 0:00:10
    --------------------------------------- 0.1/2.9 MB 653.6 kB/s eta 0:00:05
   - -------------------------------------- 0.1/2.9 MB 944.1 kB/s eta 0:00:03
   --- ------------------------------------ 0.3/2.9 MB 1.3 MB/s eta 0:00:03
   ------- -------------------------------- 0.5/2.9 MB 2.2 MB/s eta 0:00:02
   --------------- ------------------------ 1.1/2.9 MB 4.0 MB/s eta 0:00:01
   ----------------------------- ---------- 2.1/2.9 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 8.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\vaish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
# --- 1. Load Metadata and Create Initial Manifest ---
# Read the initial metadata spreadsheet
try:
    # Use the path defined in Chunk 1
    metadata = pd.read_excel(META_PATH)
    metadata["ID"] = metadata["ID"].astype(str).str.strip()
    valid_ids = set(metadata["ID"].values)
    print("✅ Metadata Loaded.")
except FileNotFoundError:
    print(f"❌ ERROR: Metadata file not found at {META_PATH}")
    raise

# Find audio files and link with metadata (similar logic to original notebook)
audio_data = []
# Folders inside 'training' are the 'Task' types (e.g., phonationA, rhythmKA)
folders = sorted(os.listdir(TRAIN_DIR))

# Process all audio files and link with metadata
for folder in tqdm(folders, desc="Linking Train Audio"):
    folder_path = os.path.join(TRAIN_DIR, folder)
    if not os.path.isdir(folder_path): continue
    for file in os.listdir(folder_path):
        if file.lower().endswith((".wav", ".mp3")): # Check for common audio extensions
            # Extract ID (e.g., 'ID000' from 'ID000_phonationA.wav')
            file_id = file.split("_")[0]
            if file_id in valid_ids:
                row = metadata[metadata["ID"] == file_id].iloc[0]
                audio_data.append({
                    "ID": file_id,
                    "Age": row["Age"],
                    "Sex": row["Sex"],
                    "Class": int(row["Class"]), # Ensure Class is integer type
                    "Task": folder,
                    # Store a relative path for portability
                    "Filepath": os.path.join(folder_path, file).replace(os.path.sep, '/')
                })

df_manifest = pd.DataFrame(audio_data)
df_manifest['Class'] = df_manifest['Class'].astype(int)

# Check the distribution and save the manifest
print(f"\n✅ Train Manifest Created (Rows: {len(df_manifest)})")
print("Class Distribution:\n", df_manifest['Class'].value_counts().sort_index())

MANIFEST_PATH = os.path.join(OUTPUT_DIR, "wav2vec_train_manifest.csv")
df_manifest.to_csv(MANIFEST_PATH, index=False)
print(f"💾 Manifest saved to: {MANIFEST_PATH}")
print("-" * 70)


# --- 2. Initialize Wav2vec 2.0 Model and Processor ---

# Load feature extractor and model from HuggingFace
# try:
#     # The processor handles loading, resampling, and padding/truncation
#     processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
#     # The model extracts the actual embeddings (the features)
#     model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
#     model.eval() # Set model to evaluation mode
#     print(f"✅ Wav2vec 2.0 Model and Processor loaded: {MODEL_NAME}")
    
#     # Check for GPU and move model if available
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
#     print(f"✅ Running on device: {device}")
    
# except Exception as e:
#     print(f"❌ ERROR: Failed to load Wav2vec 2.0 components. Ensure you have PyTorch and Transformers correctly installed.")
#     print(f"Details: {e}")
#     raise

# print("-" * 70)

try:
    # Use FeatureExtractor instead of Processor for Classification tasks
    # This avoids the error if the model has no tokenizer/vocab
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
    
    model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
    model.eval()
    
    print(f"✅ Wav2vec 2.0 Model and Feature Extractor loaded: {MODEL_NAME}")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"✅ Running on device: {device}")
    
except Exception as e:
    print(f"❌ ERROR: Failed to load components.")
    print(f"Details: {e}")
    raise

✅ Metadata Loaded.


Linking Train Audio: 100%|██████████| 8/8 [00:00<00:00, 28.09it/s]



✅ Train Manifest Created (Rows: 2176)
Class Distribution:
 Class
1     48
2    208
3    456
4    608
5    856
Name: count, dtype: int64
💾 Manifest saved to: Train/task1\wav2vec_train_manifest.csv
----------------------------------------------------------------------
✅ Wav2vec 2.0 Model and Feature Extractor loaded: facebook/wav2vec2-xls-r-300m
✅ Running on device: cpu


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [9]:
# --- 1. Define the Feature Extraction Worker Function ---
def extract_wav2vec_features(filepath, model, processor, target_sr, device):
    """
    Loads a single audio file, extracts Wav2vec 2.0 embeddings, and returns 
    the mean-pooled vector (fixed-length embedding).
    """
    import torch
    import librosa
    import numpy as np
    import os
    
    # Ensure the path is correct
    full_path = os.path.normpath(os.path.join(BASE_DIR, filepath))
    
    try:
        # Load and resample the audio to the model's expected SR (16kHz)
        speech, sr = librosa.load(full_path, sr=target_sr)

        # Preprocess the audio: convert to tensor, normalize (done by processor)
        # return_tensors="pt" gives a PyTorch tensor
        inputs = processor(
            speech, 
            sampling_rate=target_sr, 
            return_tensors="pt"
        )
        
        # Move inputs to the same device as the model (GPU or CPU)
        input_values = inputs.input_values.to(device)

        with torch.no_grad():
            # Get the hidden states from the final layer
            # We use .last_hidden_state for the contextualized embedding
            outputs = model(input_values)
            last_hidden_state = outputs.last_hidden_state.squeeze(0)
            
            # --- Mean Pooling (Aggregation) ---
            # Calculate the mean across the sequence length (axis 0) to get a 
            # single fixed-size vector for the entire audio clip.
            mean_pooled_embedding = last_hidden_state.mean(dim=0)
            
            # Convert to numpy and return
            return mean_pooled_embedding.cpu().numpy()
            
    except Exception as e:
        # Fallback to a vector of NaNs on failure
        # Size must match EMBEDDING_DIM (1024)
        # print(f"Error processing {filepath}: {e}")
        return np.full(EMBEDDING_DIM, np.nan, dtype=np.float32)

# --- 2. Main Parallel Execution ---
# Use all available cores except 1 for stability
num_cores = multiprocessing.cpu_count() - 1 if multiprocessing.cpu_count() > 1 else 1

print(f"🚀 Starting Parallel Wav2vec 2.0 Extraction on {len(df_manifest)} files using {num_cores} cores...")

# Prepare list of filepaths for parallel processing
filepaths = df_manifest['Filepath'].tolist()

# The model and processor are large objects and cannot be pickled/passed easily,
# but `joblib`'s `loky` backend handles this by passing references/spawning new processes.
# We pass the shared objects (model, processor) once.
all_embeddings = Parallel(n_jobs=num_cores, verbose=10)(
    delayed(extract_wav2vec_features)(
        fp, model, processor, TARGET_SR, device
    ) for fp in filepaths
)

# Convert the list of arrays into a single DataFrame
df_embeddings = pd.DataFrame(all_embeddings)

# --- 3. Consolidation and Cleanup ---
# Identify and handle failed/NaN rows
nan_mask = df_embeddings.isnull().all(axis=1)
num_failed = nan_mask.sum()
if num_failed > 0:
    print(f"\n⚠️ WARNING: {num_failed} files failed extraction (NaN rows). These will be removed.")
    df_manifest = df_manifest[~nan_mask].reset_index(drop=True)
    df_embeddings = df_embeddings[~nan_mask].reset_index(drop=True)

# Rename embedding columns
df_embeddings.columns = [f'wav2vec_{i}' for i in range(EMBEDDING_DIM)]

# Final merge
df_final_features = pd.concat([df_manifest[['ID', 'Class', 'Task']], df_embeddings], axis=1)

# Save the extracted features
FEATURES_PATH = os.path.join(OUTPUT_DIR, "wav2vec_audio_features.csv")
df_final_features.to_csv(FEATURES_PATH, index=False)

print(f"\n✅ Extraction complete. Final shape: {df_final_features.shape}")
print(f"💾 Features saved to: {FEATURES_PATH}")
print("-" * 70)

🚀 Starting Parallel Wav2vec 2.0 Extraction on 2176 files using 19 cores...


[Parallel(n_jobs=19)]: Using backend LokyBackend with 19 concurrent workers.


NameError: name 'processor' is not defined

In [18]:
import torch
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

# --- 1. Define the Feature Extraction Function ---
def get_embedding(filepath, model, feature_extractor, target_sr, device):
    """
    Loads audio, passes it through the model, and returns the mean-pooled embedding.
    """
    # Construct full path (assuming filepath in manifest is relative to BASE_DIR)
    # If filepath is already absolute, os.path.join handles it correctly anyway
    full_path = os.path.join(filepath)
    
    try:
        # 1. Load Audio
        speech, sr = librosa.load(full_path, sr=target_sr)

        # 2. Preprocess (Feature Extractor)
        # return_tensors="pt" creates PyTorch tensors
        inputs = feature_extractor(
            speech, 
            sampling_rate=target_sr, 
            return_tensors="pt", 
            padding=True
        )
        
        # 3. Move inputs to GPU/CPU
        input_values = inputs.input_values.to(device)

        # 4. Model Inference
        with torch.no_grad():
            outputs = model(input_values)
            # last_hidden_state shape: (Batch_Size, Sequence_Length, Hidden_Dim)
            last_hidden_state = outputs.last_hidden_state.squeeze(0)
            
            # 5. Mean Pooling
            # Average over time (axis 0) to get one vector per audio file
            mean_pooled_embedding = last_hidden_state.mean(dim=0)
            
            return mean_pooled_embedding.cpu().numpy()
            
    except Exception as e:
        print(f"⚠️ Error processing {filepath}: {e}")
        return None

# --- 2. Sequential Execution (GPU Safe) ---

print(f"🚀 Starting Wav2vec 2.0 Extraction on {len(df_manifest)} files...")
print(f"   Device: {device}")

embeddings_list = []
valid_indices = []

# We use a standard loop instead of Parallel/joblib to avoid CUDA/Pickling errors
for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="Extracting Features"):
    
    emb = get_embedding(
        row['Filepath'], 
        model, 
        feature_extractor, 
        TARGET_SR, # Ensure this variable is defined (usually 16000)
        device
    )
    
    if emb is not None:
        embeddings_list.append(emb)
        valid_indices.append(idx)

# --- 3. Consolidation and Cleanup ---

if len(embeddings_list) == 0:
    raise ValueError("❌ No embeddings were extracted. Check your file paths or audio files.")

# Convert list of arrays to DataFrame
df_embeddings = pd.DataFrame(embeddings_list)

# Determine Embedding Dimension dynamically (e.g., 768 for Base, 1024 for Large)
embedding_dim = df_embeddings.shape[1]
print(f"ℹ️ Detected Embedding Dimension: {embedding_dim}")

# Rename columns
df_embeddings.columns = [f'wav2vec_{i}' for i in range(embedding_dim)]

# Filter the original manifest to match successfully processed files
df_manifest_clean = df_manifest.iloc[valid_indices].reset_index(drop=True)

# Merge Metadata with Embeddings
df_final_features = pd.concat([df_manifest_clean[['ID', 'Class', 'Task']], df_embeddings], axis=1)

# Save the extracted features
FEATURES_PATH = os.path.join(OUTPUT_DIR, "wav2vec_audio_features.csv")
df_final_features.to_csv(FEATURES_PATH, index=False)

print(f"\n✅ Extraction complete.")
print(f"   Original Files: {len(df_manifest)}")
print(f"   Successful:     {len(df_final_features)}")
print(f"   Failed/Skipped: {len(df_manifest) - len(df_final_features)}")
print(f"💾 Features saved to: {FEATURES_PATH}")
print("-" * 70)

🚀 Starting Wav2vec 2.0 Extraction on 2176 files...
   Device: cpu


Extracting Features: 100%|██████████| 2176/2176 [8:19:22<00:00, 13.77s/it]      


ℹ️ Detected Embedding Dimension: 1024

✅ Extraction complete.
   Original Files: 2176
   Successful:     2176
   Failed/Skipped: 0
💾 Features saved to: Train/task1\wav2vec_audio_features.csv
----------------------------------------------------------------------


In [19]:
import torch
import librosa
import numpy as np
import pandas as pd
import os
from joblib import Parallel, delayed
import multiprocessing

# --- 1. Define the Worker Function ---
def extract_single_file(filepath, model, feature_extractor, target_sr, device):
    """
    Worker function to process a single audio file.
    """
    try:
        # Load audio (Librosa is not thread-safe by default, but safe enough here)
        speech, sr = librosa.load(filepath, sr=target_sr)

        # Preprocess
        inputs = feature_extractor(
            speech, 
            sampling_rate=target_sr, 
            return_tensors="pt", 
            padding=True
        )
        
        # Move to Device
        input_values = inputs.input_values.to(device)

        # Model Inference (PyTorch releases GIL here, allowing parallelism)
        with torch.no_grad():
            outputs = model(input_values)
            last_hidden_state = outputs.last_hidden_state.squeeze(0)
            mean_pooled = last_hidden_state.mean(dim=0)
            
            return mean_pooled.cpu().numpy()

    except Exception as e:
        # print(f"⚠️ Error on {filepath}: {e}") # Uncomment to debug
        return None

# --- 2. Main Parallel Execution ---

# Determine cores (leave 1 free for OS)
num_cores = multiprocessing.cpu_count() - 1 if multiprocessing.cpu_count() > 1 else 1
print(f"🚀 Starting Parallel Extraction on {len(df_manifest)} files using {num_cores} threads...")

# Prepare filepaths (Fixing the path issue: assume filepath in CSV is the full relative path)
filepaths = df_manifest['Filepath'].tolist()

# Run Parallel Loop
# prefer="threads" is CRITICAL here to share the model memory across workers
results = Parallel(n_jobs=num_cores, prefer="threads", verbose=5)(
    delayed(extract_single_file)(
        fp, model, feature_extractor, TARGET_SR, device
    ) for fp in filepaths
)

# --- 3. Consolidation & Aggregation ---

# Filter out None results (failed files)
valid_results = [r for r in results if r is not None]
valid_indices = [i for i, r in enumerate(results) if r is not None]

if not valid_results:
    raise ValueError("❌ No embeddings extracted. Check file paths.")

# Create DataFrame
df_embeddings = pd.DataFrame(valid_results)
embedding_dim = df_embeddings.shape[1]
df_embeddings.columns = [f'wav2vec_{i}' for i in range(embedding_dim)]

# Filter Metadata
df_manifest_clean = df_manifest.iloc[valid_indices].reset_index(drop=True)

# Merge
df_final_features = pd.concat([df_manifest_clean[['ID', 'Class', 'Task']], df_embeddings], axis=1)

print(f"\n✅ Extraction Complete. Processed: {len(df_final_features)}")

# --- 4. Aggregation (Subject Level) ---
print("🔄 Aggregating by Subject ID...")
aggregation_dict = {
    'Class': 'first',
    **{col: 'mean' for col in df_embeddings.columns}
}

df_master = df_final_features.drop(columns=['Task']).groupby('ID').agg(aggregation_dict).reset_index()

# Save
FEATURES_PATH = os.path.join(OUTPUT_DIR, "Task1_WAV2VEC_MASTER_FEATURES.csv")
df_master.to_csv(FEATURES_PATH, index=False)
print(f"💾 Saved Master Features to: {FEATURES_PATH}")

🚀 Starting Parallel Extraction on 2176 files using 19 threads...


[Parallel(n_jobs=19)]: Using backend ThreadingBackend with 19 concurrent workers.
[Parallel(n_jobs=19)]: Done  34 tasks      | elapsed:   58.4s
[Parallel(n_jobs=19)]: Done 124 tasks      | elapsed:  2.7min
[Parallel(n_jobs=19)]: Done 250 tasks      | elapsed:  5.2min
[Parallel(n_jobs=19)]: Done 412 tasks      | elapsed:  8.1min
[Parallel(n_jobs=19)]: Done 610 tasks      | elapsed: 11.8min
[Parallel(n_jobs=19)]: Done 844 tasks      | elapsed: 16.3min
[Parallel(n_jobs=19)]: Done 1114 tasks      | elapsed: 21.5min
[Parallel(n_jobs=19)]: Done 1420 tasks      | elapsed: 27.2min
[Parallel(n_jobs=19)]: Done 1762 tasks      | elapsed: 34.2min
[Parallel(n_jobs=19)]: Done 2176 out of 2176 | elapsed: 42.9min finished



✅ Extraction Complete. Processed: 2176
🔄 Aggregating by Subject ID...
💾 Saved Master Features to: Train/task1\Task1_WAV2VEC_MASTER_FEATURES.csv


In [21]:
!pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB 435.7 kB/s eta 0:02:46
   ---------------------------------------- 0.1/72.0 MB 1.1 MB/s eta 0:01:06
   ---------------------------------------- 0.3/72.0 MB 1.7 MB/s eta 0:00:42
   ---------------------------------------- 0.5/72.0 MB 2.8 MB/s eta 0:00:26
    --------------------------------------- 1.1/72.0 MB 4.5 MB/s eta 0:00:16
   - -------------------------------------- 2.5/72.0 MB 9.0 MB/s eta 0:00:08
   -- ------------------------------------- 3.7/72.0 MB 11.9 MB/s eta 0:00:06
   --- ------------------------------------ 5.9/72.0 MB 16.4 MB/s eta 0:00:05
   ---- ----------------------------------- 8.4/72.0 MB 20.6 MB/s eta 0:00:04
   ----- ---------------------------------- 10.3/72.0 MB 28.4 MB/s eta 0:00:03
   ------- -------------------------------- 13.0/72.0 MB 54.7 MB/s eta 0:00:02
   -


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\vaish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [23]:
# --- 1. Import necessary libraries for ML ---
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, make_scorer
import pandas as pd
import numpy as np
import os

# --- 2. Configuration ---
FEATURES_PATH = os.path.join(BASE_DIR, "Task1_WAV2VEC_MASTER_FEATURES.csv")
N_SPLITS = 5
RANDOM_STATE = 42

# --- 3. Load Data ---
try:
    df_master = pd.read_csv(FEATURES_PATH)
    if df_master.shape[0] == 0:
        raise ValueError("Master features DataFrame is empty.")
    print(f"✅ Loaded Master Features. Shape: {df_master.shape}")
except FileNotFoundError:
    print(f"❌ ERROR: Master feature file not found at {FEATURES_PATH}. Check Chunk 3 output.")
    raise
except ValueError as e:
    print(f"❌ ERROR: {e}")
    raise

# --- 4. Prepare X and y ---
X = df_master.drop(columns=['ID', 'Class'])
# Convert Class 1-5 to 0-4 for XGBoost (required for multi-class classification)
y = df_master['Class'] - 1

print(f"Class Distribution in y (0-4): \n{y.value_counts().sort_index()}")

# --- 5. Feature Scaling ---
# Scale the high-dimensional Wav2vec 2.0 embeddings
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("✅ Features scaled using RobustScaler.")

# --- 6. Model Definition ---
# Use parameters suitable for a high-dimensional feature set
xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=2,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
    tree_method='hist' # Efficient tree method
)

# --- 7. K-Fold Cross-Validation ---
# Create a custom scorer for Macro F1-score
macro_f1_scorer = make_scorer(f1_score, average='macro')

# Stratified split is crucial for class imbalance
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f"\n🚀 Starting {N_SPLITS}-Fold Stratified Cross-Validation (Metric: Macro F1-score)...")

# Perform cross-validation
cv_scores = cross_val_score(
    xgb_model, 
    X_scaled, # Use scaled features for improved performance
    y, 
    cv=cv, 
    scoring=macro_f1_scorer, 
    n_jobs=-1, # Use all available cores for CV folds
    verbose=1
)

# --- 8. Results Summary ---
print("\n--- Cross-Validation Results (Wav2vec 2.0 + XGBoost) ---")
print(f"Individual Fold Macro F1 Scores: {np.round(cv_scores, 4)}")
print(f"Average Macro F1 Score: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
print("-" * 70)

✅ Loaded Master Features. Shape: (272, 1026)
Class Distribution in y (0-4): 
Class
0      6
1     26
2     57
3     76
4    107
Name: count, dtype: int64
✅ Features scaled using RobustScaler.

🚀 Starting 5-Fold Stratified Cross-Validation (Metric: Macro F1-score)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   23.7s remaining:   35.6s



--- Cross-Validation Results (Wav2vec 2.0 + XGBoost) ---
Individual Fold Macro F1 Scores: [0.3592 0.247  0.249  0.3561 0.4604]
Average Macro F1 Score: 0.3343 (+/- 0.0799)
----------------------------------------------------------------------


[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   24.3s finished


In [26]:
!pip install imblearn

   ---------------------------------------- 0.0/240.0 kB ? eta -:--:--
   ----- ---------------------------------- 30.7/240.0 kB 1.3 MB/s eta 0:00:01
   --------------- ------------------------ 92.2/240.0 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 240.0/240.0 kB 2.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\vaish\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import f1_score, make_scorer, confusion_matrix, classification_report
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler

# --- Configuration ---
# Update this to your actual path if needed
BASE_DIR = r"Train/task1"
FEATURES_PATH = os.path.join(BASE_DIR, "Task1_WAV2VEC_MASTER_FEATURES.csv")
N_SPLITS = 5
RANDOM_STATE = 42

# --- 1. Load Data ---
if not os.path.exists(FEATURES_PATH):
    print(f"❌ Error: File not found at {FEATURES_PATH}")
else:
    df_master = pd.read_csv(FEATURES_PATH)
    print(f"✅ Data Loaded. Shape: {df_master.shape}")

    # Prepare X and y
    X = df_master.drop(columns=['ID', 'Class'])
    # Ensure y is 0-4 for XGBoost
    le = LabelEncoder()
    y = le.fit_transform(df_master['Class'])
    
    # Check for the minority class size to safely configure SMOTE
    min_class_size = np.min(np.bincount(y))
    print(f"ℹ️ Minority class has {min_class_size} samples.")

    # --- 2. Define Pipelines ---

    # PIPELINE 1: Improved XGBoost
    # - PCA: Reduces 1024 features to ~50-100 essential components
    # - RandomOverSampler: Safe for tiny classes (like Class 1 with 6 samples) where SMOTE fails
    xgb_pipeline = ImbPipeline([
        ('scaler', RobustScaler()),
        ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)), # Keep 95% variance
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)), # Duplicate minority samples
        ('classifier', XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.6,
            eval_metric='mlogloss',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    # PIPELINE 2: Support Vector Machine (SVM)
    # - SVMs are often superior for High-Dimension/Low-Sample Count tasks
    # - 'class_weight="balanced"' handles imbalance automatically
    svm_pipeline = ImbPipeline([
        ('scaler', RobustScaler()),
        ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
        ('classifier', SVC(
            kernel='rbf',
            C=10.0,          # Regularization (Higher = stricter)
            gamma='scale', 
            class_weight='balanced',
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])

    # --- 3. Evaluation Loop ---
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scorer = make_scorer(f1_score, average='macro')

    print(f"\n🚀 Running {N_SPLITS}-Fold CV on {len(X)} samples...")

    # Evaluate XGBoost
    print("\n--- 1. XGBoost with PCA & Oversampling ---")
    xgb_scores = cross_val_score(xgb_pipeline, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    print(f"Scores: {np.round(xgb_scores, 4)}")
    print(f"🔥 Average Macro F1: {np.mean(xgb_scores):.4f} (+/- {np.std(xgb_scores):.4f})")

    # Evaluate SVM
    print("\n--- 2. SVM with PCA & Oversampling ---")
    svm_scores = cross_val_score(svm_pipeline, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    print(f"Scores: {np.round(svm_scores, 4)}")
    print(f"🔥 Average Macro F1: {np.mean(svm_scores):.4f} (+/- {np.std(svm_scores):.4f})")

    # --- 4. Detailed Report on Best Model (e.g., SVM) ---
    print("\n📊 Generating detailed report for SVM (on one split)...")
    train_idx, test_idx = next(cv.split(X, y))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    svm_pipeline.fit(X_train, y_train)
    y_pred = svm_pipeline.predict(X_test)
    
    # Decode labels back to 1-5
    y_test_orig = le.inverse_transform(y_test)
    y_pred_orig = le.inverse_transform(y_pred)
    
    print(classification_report(y_test_orig, y_pred_orig, digits=4))

✅ Data Loaded. Shape: (272, 1026)
ℹ️ Minority class has 6 samples.

🚀 Running 5-Fold CV on 272 samples...

--- 1. XGBoost with PCA & Oversampling ---
Scores: [0.3561 0.2938 0.1996 0.3003 0.5782]
🔥 Average Macro F1: 0.3456 (+/- 0.1267)

--- 2. SVM with PCA & Oversampling ---
Scores: [0.3938 0.2992 0.2413 0.3946 0.4323]
🔥 Average Macro F1: 0.3523 (+/- 0.0708)

📊 Generating detailed report for SVM (on one split)...
              precision    recall  f1-score   support

           1     0.0000    0.0000    0.0000         1
           2     0.6000    0.5000    0.5455         6
           3     0.4000    0.3636    0.3810        11
           4     0.4000    0.5333    0.4571        15
           5     0.6316    0.5455    0.5854        22

    accuracy                         0.4909        55
   macro avg     0.4063    0.3885    0.3938        55
weighted avg     0.5072    0.4909    0.4945        55



In [28]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, make_scorer, classification_report

# --- Configuration ---
BASE_DIR = r"Train/task1"
PATH_HANDCRAFTED = os.path.join(BASE_DIR, "Task1_MASTER_FEATURES.csv")        # From Step 3
PATH_WAV2VEC = os.path.join(BASE_DIR, "Task1_WAV2VEC_MASTER_FEATURES.csv")    # From Step 4
RANDOM_STATE = 42
N_SPLITS = 5

def load_and_fuse_data():
    print("🔄 Loading Datasets...")
    
    # 1. Load datasets
    if not os.path.exists(PATH_HANDCRAFTED) or not os.path.exists(PATH_WAV2VEC):
        print("❌ Error: Missing one of the feature files.")
        return None, None

    df_hand = pd.read_csv(PATH_HANDCRAFTED)
    df_deep = pd.read_csv(PATH_WAV2VEC)
    
    print(f"   Clinical Features Shape: {df_hand.shape}")
    print(f"   Wav2vec Features Shape:  {df_deep.shape}")

    # 2. Merge on ID (and Class/Task if present to ensure alignment)
    # We drop 'Class' from one df to avoid duplication
    df_deep_clean = df_deep.drop(columns=['Class', 'Task'], errors='ignore')
    
    df_fused = pd.merge(df_hand, df_deep_clean, on='ID', how='inner')
    print(f"✅ Fused Dataset Shape: {df_fused.shape}")
    
    # 3. Prepare X and y
    # Drop non-feature columns
    cols_to_drop = ['ID', 'Class', 'Task']
    X = df_fused.drop(columns=[c for c in cols_to_drop if c in df_fused.columns])
    
    # Identify column types for separate processing (optional, but good for debugging)
    feat_cols = X.columns.tolist()
    
    # Prepare Labels
    le = LabelEncoder()
    y = le.fit_transform(df_fused['Class'])
    
    return X, y, le

# --- Main Execution ---
X, y, le = load_and_fuse_data()

if X is not None:
    print("\n🚀 Building Ensemble Classifier (SVM + RF + XGB)...")
    
    # --- Individual Classifiers ---
    
    # 1. SVM (Great for high dimensions)
    clf_svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
    
    # 2. Random Forest (Robust to noise)
    clf_rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=RANDOM_STATE)
    
    # 3. XGBoost (Gradient Boosting)
    clf_xgb = XGBClassifier(
        n_estimators=200, 
        learning_rate=0.1, 
        max_depth=4, 
        subsample=0.8,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    # --- The Voting Ensemble ---
    # Combines the probabilities of all three models
    voting_clf = VotingClassifier(
        estimators=[('svm', clf_svm), ('rf', clf_rf), ('xgb', clf_xgb)],
        voting='soft',
        n_jobs=-1
    )

    # --- The Full Pipeline ---
    # 1. Scale Data
    # 2. Reduce Dimensions (PCA) - Vital to compress the 1000+ features
    # 3. Oversample (Balance the classes)
    # 4. Classify (Voting Ensemble)
    final_pipeline = ImbPipeline([
        ('scaler', RobustScaler()),
        ('pca', PCA(n_components=0.98, random_state=RANDOM_STATE)), # Keep 98% variance
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
        ('voting', voting_clf)
    ])

    # --- Cross-Validation ---
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scorer = make_scorer(f1_score, average='macro')
    
    print(f"🔄 Running {N_SPLITS}-Fold Cross-Validation...")
    scores = cross_val_score(final_pipeline, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    
    print("\n" + "="*50)
    print(f"🔥 FINAL FUSION RESULTS")
    print("="*50)
    print(f"Individual Folds: {np.round(scores, 4)}")
    print(f"AVERAGE MACRO F1: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
    print("="*50)
    
    # Detailed report on one split
    print("\n📊 Detailed Classification Report (Sample Split):")
    train_idx, test_idx = next(cv.split(X, y))
    final_pipeline.fit(X.iloc[train_idx], y[train_idx])
    y_pred = final_pipeline.predict(X.iloc[test_idx])
    print(classification_report(y[test_idx], y_pred, target_names=[str(c) for c in le.classes_]))

🔄 Loading Datasets...
   Clinical Features Shape: (272, 40)
   Wav2vec Features Shape:  (272, 1026)
✅ Fused Dataset Shape: (272, 1064)

🚀 Building Ensemble Classifier (SVM + RF + XGB)...
🔄 Running 5-Fold Cross-Validation...

🔥 FINAL FUSION RESULTS
Individual Folds: [0.4588 0.305  0.3391 0.4615 0.4511]
AVERAGE MACRO F1: 0.4031 (+/- 0.0671)

📊 Detailed Classification Report (Sample Split):
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.75      0.50      0.60         6
           3       0.55      0.55      0.55        11
           4       0.47      0.60      0.53        15
           5       0.65      0.59      0.62        22

    accuracy                           0.56        55
   macro avg       0.48      0.45      0.46        55
weighted avg       0.58      0.56      0.57        55

